In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data_path = 'DataFolder/train.csv'
df = pd.read_csv(data_path)
df.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\r\n\r\nDOWNL...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \r\n🚨Straight Outta Cross Keys SC 🚨You...,[15 Amazing Hidden Features Of Google Search Y...,0
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \r...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1


In [71]:
import torch
from torch.utils.data import Dataset

class RedditRulesDataset(Dataset):

    def __init__(self, df, tokenizer, max_seq_length=128, data_dtype=torch.float32):
        self.df = df
        self.tokenizer = tokenizer

        texts = (
            "RULE: " + df["rule"].astype(str) + "\n"
            "SUBREDDIT: " + df["subreddit"].astype(str) + "\n"
            "POS: " + (df["positive_example_1"].astype(str).fillna("")) + " || " +
                      (df["positive_example_2"].astype(str).fillna("")) + "\n"
            "NEG: " + (df["negative_example_1"].astype(str).fillna("")) + " || " +
                      (df["negative_example_2"].astype(str).fillna("")) + "\n"
            "COMMENT: " + df["body"].astype(str)
        ).tolist()

        enc = tokenizer(
            texts,
            truncation=True,
            max_length=max_seq_length,
            padding=True,
        )

        self.input_ids = torch.tensor(enc["input_ids"], dtype=data_dtype)
        self.attention_mask = torch.tensor(enc["attention_mask"], dtype=data_dtype)

        self.labels = torch.tensor(df["rule_violation"].values, dtype=data_dtype)

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

class PairRedditRulesDataset(Dataset):

    def __init__(self, df, tokenizer, max_seq_length=128, data_dtype=torch.float32):
        self.df = df
        self.tokenizer = tokenizer

        comment_text = df["body"].astype(str).tolist()
        rule_context_text = (
            "RULE: " + df["rule"].astype(str) + "\n"
            "SUBREDDIT: " + df["subreddit"].astype(str) + "\n"
            "POS: " + (df["positive_example_1"].astype(str).fillna("")) + " || " +
                      (df["positive_example_2"].astype(str).fillna("")) + "\n"
            "NEG: " + (df["negative_example_1"].astype(str).fillna("")) + " || " +
                      (df["negative_example_2"].astype(str).fillna(""))
        ).tolist()

        enc = tokenizer(
            text=comment_text,
            text_pair = rule_context_text,
            truncation=True,
            max_length=max_seq_length,
            padding=True,
        )

        self.input_ids = torch.tensor(enc["input_ids"], dtype=data_dtype)
        self.attention_mask = torch.tensor(enc["attention_mask"], dtype=data_dtype)

        self.labels = torch.tensor(df["rule_violation"].values, dtype=data_dtype)

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }


In [76]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
ds = RedditRulesDataset(df, tokenizer, max_seq_length=384, data_dtype=torch.float64)
pairds = PairRedditRulesDataset(df, tokenizer, max_seq_length=384, data_dtype=torch.float64)
collate = DataCollatorWithPadding(tokenizer)

loader = DataLoader(ds, batch_size=32, shuffle=True, collate_fn=collate)
pairloader = DataLoader(pairds, batch_size=32, shuffle=True, collate_fn=collate)

batch = next(iter(pairloader))

In [77]:
print(batch["input_ids"][0])
print(batch["input_ids"][0].dtype)

tensor([  101.,  2123.,  1005.,  1056.,  2031.,  2151.,  3538.,  1997.,  6040.,
         2021.,  2000.,  5959.,  2009., 12043.,  1012.,  2030.,  2017.,  2071.,
         2074.,  3046.,  3287., 11598.,  2869.,  2074.,  2066.,  1045.,  2106.,
         1012.,  1045.,  2145.,  2031.,  1037.,  2261.,  2489., 19575.,  2098.,
         8648.,  5644.,  1012.,  3246.,  2023.,  7126.,  2017.,  2004.,  2009.,
         2106.,  2000.,  2033.,   999., 16770.,  1024.,  1013.,  1013.,  7479.,
         1012., 12391.,  3527., 16761.,  1012.,  4012.,  1013.,  5950.,  1012.,
        25718.,  1029.,  1037.,  1035.,  8909.,  1027.,  3156., 22022.,  2620.,
         1004.,  1037.,  1035., 18133.,  2078.,  1027., 15263.,  2620.,  1004.,
        18133.,  2078.,  1027.,  1015.,   102.,  3627.,  1024.,  2053.,  6475.,
         1024., 12403.,  2213.,  1010.,  6523.,  7941.,  6971.,  1010.,  4895.,
        19454., 28775.,  3064.,  6475.,  1010.,  1998., 10319.,  4180.,  2024.,
         2025.,  3039.,  1012.,  4942., 

In [64]:
print(df.iloc[10])
print(ds[10])
vocab = tokenizer.get_vocab()
print({k: v for k, v in vocab.items() if k in ['[CLS]', '[SEP]', '[PAD]', '[UNK]', '[MASK]']})

unused_count = 0
for k, v in vocab.items():
    if k.startswith('[unused'):
        unused_count += 1

print(f"Number of unused tokens in the tokenizer vocabulary: {unused_count} out of {len(vocab)} total tokens")

row_id                                                               10
body                  They don't want you taking Jimmy Jonathan's to...
rule                  No legal advice: Do not offer or request legal...
subreddit                                                      politics
positive_example_1    Get a Glock 19 handgun.  Learn how to use it a...
positive_example_2    Maybe, but it still is illegal for collections...
negative_example_1    Oh, you can buy the abortion pill online for m...
negative_example_2    > how do you retaliate against them?\r\n\r\nYo...
rule_violation                                                        1
Name: 10, dtype: object
{'input_ids': tensor([  101.,  3627.,  1024.,  2053.,  3423.,  6040.,  1024.,  2079.,  2025.,
         3749.,  2030.,  5227.,  3423.,  6040.,  1012.,  4942.,  5596., 23194.,
         1024.,  4331., 13433.,  2015.,  1024.,  2131.,  1037.,  1043.,  7878.,
         2539., 28497.,  1012.,  4553.,  2129.,  2000.,  2224.,  2009.,  2

In [70]:
### Tokenizer test ###
test_text = "Hej jag heter Gabriel RULE: || COMMENT: POS: NEG: SUBREDDIT: \n"
test_context = "RULE: hellooo"

enc = tokenizer(
            test_text,
            truncation=True,
            max_length=40,
            padding="max_length"
        )

pairenc = tokenizer(
            text = test_text,
            text_pair = test_context,
            truncation=True,
            max_length=40,
            padding="max_length"
        )
print(pairenc["input_ids"])
for id in pairenc["input_ids"]:
    for k, v in vocab.items():
        if v == id:
            print(f"{id}: {k}")
    

[101, 2002, 3501, 14855, 2290, 21770, 2121, 6127, 3627, 1024, 1064, 1064, 7615, 1024, 13433, 2015, 1024, 11265, 2290, 1024, 4942, 5596, 23194, 1024, 102, 3627, 1024, 7592, 9541, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
101: [CLS]
2002: he
3501: ##j
14855: ja
2290: ##g
21770: het
2121: ##er
6127: gabriel
3627: rule
1024: :
1064: |
1064: |
7615: comment
1024: :
13433: po
2015: ##s
1024: :
11265: ne
2290: ##g
1024: :
4942: sub
5596: ##red
23194: ##dit
1024: :
102: [SEP]
3627: rule
1024: :
7592: hello
9541: ##oo
102: [SEP]
0: [PAD]
0: [PAD]
0: [PAD]
0: [PAD]
0: [PAD]
0: [PAD]
0: [PAD]
0: [PAD]
0: [PAD]
0: [PAD]
